# Local gNFW constraints with the saved MOPED NPE

This notebook generates **one new masked SO baseline-deproj0 cross-spectrum** for nine user-selected pressure parameters, then uses the **existing trained MOPED estimator**. The fiducial defaults are Battaglia12. No model or compression is refitted.

Pipeline: existing HalfDome Julia simulator -> already-beamed, masked noisy C_ell -> training-contract signed binned D_ell -> saved asinh/standardization -> saved MOPED projection/standardization -> prior-restricted NPE samples -> corner plot.

The halo catalogue, cosmology, 2 arcmin beam, mask seed 12345, mask apodization and noise conventions must match training. **Noise-contract correction:** the legacy training simulations reused one noise realization across pressure rows. Independent noise changes the observation distribution, not just the pressure truth. The completed [fixed-noise diagnostic notebook](moped_fixed_noise_diagnostic.ipynb) compares both observations and displays the conditional corner plot. A single truth-containing contour is **not SBC or proof of coverage**.

**Verification limit:** the local Julia environment and all nine parameter inputs passed a constructor check, and the NPE passed cluster/local log-probability checks. The exact historical XGPaint source revision used for the 524k simulations has not yet been independently matched. Before treating new-profile constraints as an accuracy validation, compare that source snapshot or reproduce a known training-row signal. The notebook records the actual library source hashes; it does not silently claim historical equivalence.

Full nside=4096 map generation is expensive and memory intensive. The cluster wrapper requested 128 GB; this WSL machine currently exposes about 31 GiB. Do not reduce nside or change beam to make this fit: that changes the model's observable. Set the Julia executable and sufficient RAM before enabling generation. No FITS files are read by the Python analysis.


In [1]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

HERE = Path.cwd().resolve()
REPO = next((p for p in [HERE, *HERE.parents] if (p / "SBI_analysis/so_moped_local.py").is_file()), None)
if REPO is None:
    raise FileNotFoundError("Open this notebook inside the HalfDome checkout.")
sys.path.insert(0, str(REPO / "SBI_analysis"))
import importlib
import so_moped_local
importlib.reload(so_moped_local)
from so_moped_local import (
    load_bundle, truth_vector, simulator_command, simulate, prepare_context,
    load_checked_model, posterior_samples, plot_corner,
)
from so_sbi_compression import PARAM_NAMES, project, save_npz, write_json, metrics_from_samples

BUNDLE = REPO / "SBI_analysis/outputs/compression_bestval_20260909/moped_bundle"
contract, transform, bundle_manifest = load_bundle(BUNDLE)
print("Experiment:", bundle_manifest["experiment_id"])
print("Saved transform round-trip: passed")


Experiment: 35fbe0a27078c4b34d8aac9aee2b95fb4e2e4c977dc720ac516dd91176c302d6
Saved transform round-trip: passed


## Parameters and simulator settings
Edit any of the nine entries below within the **saved inference prior**. Keep the other simulator settings fixed. The legacy training generator reused noise seed 12345 across parameter rows. An independently selected noise seed is a distribution-shift test, not a matched conditional observation. The fixed-noise diagnostic below preserves your original settings and observation.

`RUN_SIMULATION=False` avoids launching a large map calculation accidentally. Once Julia and RAM are ready, set it to `True`. Completed observations are reused only when the request and source-code hashes match. A changed parameter or seed gets its own directory; no training file is overwritten.


In [11]:
PRESSURE_PARAMS = {
    "P0": 2,
    "xc": 0.8,
    "beta": 5.1,
    "alpha_m_P0": 0.01,
    "alpha_m_xc": -0.08,
    "alpha_m_beta": 0.07,
    "alpha_z_P0": -1.3,
    "alpha_z_xc": 0.16,
    "alpha_z_beta": 0.09,
}

CATALOGUE = REPO / "lightcone_100.hdf5"
NOISE_CURVE = REPO / "other_sims/SO/SO_LAT_Nell_T_atmv1_baseline_fsky0p4_ILC_tSZ.txt"
JULIA = os.environ.get("JULIA", "/home/kn18001/.julia/juliaup/julia-1.12.2+0.x64.linux.gnu/bin/julia")
JULIA_PROJECT = Path("/home/kn18001/.julia/environments/v1.12")
JULIA_DEPOT = Path("/home/kn18001/.julia")
THREADS = 128
MASK_SEED = 12345
NOISE_SEED = 12345
RUN_SIMULATION = True

NUM_SAMPLES = 10000
MAX_PROPOSALS = 1000000
SAMPLING_SECONDS = 300
SAMPLING_SEED = 47
PILOT_SAMPLES = 20000

theta_true = truth_vector(PRESSURE_PARAMS, contract)
request_id = hashlib.sha256(json.dumps(
    {"theta": theta_true.tolist(), "mask_seed": MASK_SEED, "noise_seed": NOISE_SEED},
    sort_keys=True,
).encode()).hexdigest()[:16]
OUTPUT_DIR = REPO / "SBI_analysis/outputs/moped_local_profiles" / request_id
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
display(pd.DataFrame({"parameter": PARAM_NAMES, "truth": theta_true,
                      "prior_low": contract["low"], "prior_high": contract["high"]}))
print("Catalogue found:", CATALOGUE.is_file())
print("Noise curve found:", NOISE_CURVE.is_file())
print("Julia executable:", shutil.which(str(JULIA)))
print("Output:", OUTPUT_DIR)


,parameter,truth,prior_low,prior_high
0,P0,2.00,1.832524,34.341221
1,xc,0.80,0.150011,0.844503
2,beta,5.10,3.480627,5.216611
3,alpha_m_P0,0.01,0.000312,0.292251
4,alpha_m_xc,-0.08,-0.099718,0.099795
5,alpha_m_beta,0.07,-0.019935,0.099767
6,alpha_z_P0,-1.30,-1.363457,-0.228839
7,alpha_z_xc,0.16,0.147393,1.314474
8,alpha_z_beta,0.09,0.083808,0.745884


Catalogue found: True
Noise curve found: True
Julia executable: /home/kn18001/.julia/juliaup/julia-1.12.2+0.x64.linux.gnu/bin/julia
Output: /home/cbllover/HalfDome/SBI_analysis/outputs/moped_local_profiles/8f3cf4808a57f1f8


In [12]:
command = simulator_command(
    REPO, OUTPUT_DIR, CATALOGUE, NOISE_CURVE, theta_true,
    julia=JULIA, threads=THREADS, mask_seed=MASK_SEED, noise_seed=NOISE_SEED,
    julia_project=JULIA_PROJECT, julia_depot=JULIA_DEPOT,
)
print("Simulator command:")
import shlex
print("JULIA_DEPOT_PATH=" + shlex.quote(str(JULIA_DEPOT)) + " " + shlex.join(command))
if RUN_SIMULATION or (OUTPUT_DIR / "simulation_complete.json").is_file():
    raw_path = simulate(
        REPO, OUTPUT_DIR, CATALOGUE, NOISE_CURVE, theta_true,
        julia=JULIA, threads=THREADS, mask_seed=MASK_SEED, noise_seed=NOISE_SEED,
        julia_project=JULIA_PROJECT, julia_depot=JULIA_DEPOT,
    )
else:
    raise RuntimeError(
        "No completed matched simulation for these parameters. Configure Julia and sufficient RAM, "
        "then set RUN_SIMULATION=True. This notebook will not substitute an unrelated saved profile."
    )


Simulator command:
JULIA_DEPOT_PATH=/home/kn18001/.julia /home/kn18001/.julia/juliaup/julia-1.12.2+0.x64.linux.gnu/bin/julia --project=/home/kn18001/.julia/environments/v1.12 --threads=128 --startup-file=no /home/cbllover/HalfDome/tSZ_visuals/run_halfdome_fullsky_so_noise.jl catalog_source=halfdome simulation_name=halfdome_lightcone_100 batching_mode=full halfdome_path=/home/cbllover/HalfDome/lightcone_100.hdf5 apply_mass_cut=true mass_min=1000000000000.0 cosmo_h=0.68 cosmo_omegab=0.049 cosmo_omegac=0.261 sobol_row=0 sobol_csv_path= cleanup_nonpositive_profile_values=true output_dir=/home/cbllover/HalfDome/SBI_analysis/outputs/moped_local_profiles/8f3cf4808a57f1f8/raw cache_dir=/home/cbllover/HalfDome/SBI_analysis/outputs/moped_local_profiles/8f3cf4808a57f1f8/cache baseline_noise_path=/home/cbllover/HalfDome/other_sims/SO/SO_LAT_Nell_T_atmv1_baseline_fsky0p4_ILC_tSZ.txt goal_noise_path=/home/cbllover/HalfDome/other_sims/SO/SO_LAT_Nell_T_atmv1_goal_fsky0p4_ILC_tSZ.txt nside=4096 cl_lmax

Computing cross C_l with lmax=7979, niter=0.
Saved masked_baseline_noise_cross C_l NumPy array to /home/cbllover/HalfDome/SBI_analysis/outputs/moped_local_profiles/8f3cf4808a57f1f8/raw/halfdome_fullsky_masked_baseline_noise_cross_cl_m200c_nside4096_base_plus_battaglia_P0_amp_2p0__battaglia_P0_alpha_m_0p01__battaglia_P0_alpha_z_m1p3__battaglia_x_c_amp_0p8__battaglia_x_c_alpha_m_m0p08__batt_haac335e9e40b.npy
Finished full-sky HalfDome SO-noise run in 593.62 s.


ValueError: /home/cbllover/HalfDome/SBI_analysis/outputs/moped_local_profiles/8f3cf4808a57f1f8/raw/halfdome_fullsky_masked_baseline_noise_cross_cl_m200c_nside4096_base_plus_battaglia_P0_amp_2p0__battaglia_P0_alpha_m_0p01__battaglia_P0_alpha_z_m1p3__battaglia_x_c_amp_0p8__battaglia_x_c_alpha_m_m0p08__batt_haac335e9e40b.npy does not encode the required beam/mask/seed/deprojection configuration. Missing filename patterns: ['gaussbeam_2(?:p0+)?arcmin', 'so_fsky0p4(?:0*)?_apo60(?:p0+)?arcmin', 'deproj0', 'lmax7979', '(?:seed12345|maskseed12345_noiseseed12345)']

## Observation and preprocessing checks
The weights and bin boundaries come from the exported prepared-dataset contract, not a new Delta-ell rule. The beam and mask are already in the spectrum: **neither is applied again**. Negative cross-spectrum bins are allowed.

Background spectra are 512 saved optimization-row examples, not independent validation observations. Their range is only a rough support diagnostic. A curve falling within each separate bin's range does not guarantee joint support.


In [ ]:
x_obs, context = prepare_context(raw_path, contract, transform)
save_npz(OUTPUT_DIR / "observation.npz",
         x=x_obs, context=context, theta=theta_true,
         param_names=contract["param_names"], ell_binned=contract["ell_binned"],
         raw_profile=np.asarray(str(raw_path)), experiment_id=contract["experiment_id"])

plt.rcParams.update({"font.family": "serif", "mathtext.fontset": "cm", "font.size": 10})
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ell = contract["ell_binned"]
reference = contract["reference_x"]
for row in reference[:60]:
    axes[0].plot(ell, row, color="0.55", alpha=.12, lw=.6)
lo, med, hi = np.quantile(reference, [.01, .5, .99], axis=0)
axes[0].fill_between(ell, lo, hi, alpha=.18, color="#4477aa", label="Training examples: 1-99%")
axes[0].plot(ell, med, color="#4477aa", lw=1, label="Training examples: median")
axes[0].plot(ell, x_obs, color="#cc3311", lw=1.5, label="Selected noisy observation")
axes[0].set_xlabel(r"$\ell$")
axes[0].set_ylabel(r"$D_\ell^{yy}$")
threshold = max(float(np.median(np.abs(reference))) * .01, 1e-30)
axes[0].set_yscale("symlog", linthresh=threshold)
axes[0].legend(fontsize=8)
ref_context = contract["reference_context"]
qlo, qmed, qhi = np.quantile(ref_context, [.01, .5, .99], axis=0)
components = np.arange(1, len(context) + 1)
axes[1].fill_between(components, qlo, qhi, alpha=.2, color="#4477aa", label="Training examples: 1-99%")
axes[1].plot(components, qmed, color="#4477aa")
axes[1].plot(components, context, "o-", color="#cc3311", label="Saved-transform observation")
axes[1].set_xlabel("MOPED component")
axes[1].set_ylabel("Standardized compressed coordinate")
axes[1].legend(fontsize=8)
for ax in axes:
    ax.grid(alpha=.2)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "observation_vs_training.png", dpi=200)
plt.show()
display(pd.DataFrame({"component": components, "observation": context,
                      "reference_q01": qlo, "reference_q99": qhi,
                      "outside_reference_98pct": (context < qlo) | (context > qhi)}))
print("Signed D_ell shape:", x_obs.shape, "MOPED shape:", context.shape)


## Load and sample the trained estimator
Before sampling, 32 fixed log-probability evaluations must agree with the cluster. This tests deserialization, conditioning and the internal parameter scaling.

The saved estimator is an SBI 0.22 nflows Flow. Direct calls avoid passing it into an incompatible newer DirectPosterior wrapper. Samples are retained only inside the saved nine-dimensional BoxUniform prior, which implements the same prior restriction as direct posterior sampling. No additional theta rescaling, boundary clipping or MCMC fallback is used. Sampling has finite proposal/time budgets; a budget failure is a diagnostic, not a posterior.

A high accepted fraction does not establish calibration. The model must still be evaluated on independent held-out observations.


In [ ]:
import torch
torch.set_num_threads(min(THREADS, 8))
density_estimator = load_checked_model(BUNDLE, contract)
print("Cluster/local log-probability comparison: passed")
# No inference.pkl is needed; the learned flow contains its internal theta transform.
samples, proposals, acceptance = posterior_samples(
    density_estimator, context, contract,
    count=NUM_SAMPLES, seed=SAMPLING_SEED,
    max_proposals=MAX_PROPOSALS, seconds=SAMPLING_SECONDS,
    pilot_count=PILOT_SAMPLES, diagnostics_dir=OUTPUT_DIR / "diagnostics",
)
np.save(OUTPUT_DIR / "posterior_samples.npy", samples)
metric = metrics_from_samples(samples, theta_true, contract["low"], contract["high"])
write_json(OUTPUT_DIR / "sampling_summary.json", {
    "experiment_id": bundle_manifest["experiment_id"],
    "theta_true": theta_true.tolist(),
    "n_samples": len(samples), "proposals": proposals,
    "raw_prior_acceptance": acceptance, "seed": SAMPLING_SEED,
    "raw_profile": str(raw_path),
    "warning": "One observation is not a calibration or SBC test.",
})
print(f"Accepted prior fraction: {acceptance:.3%}")
display(pd.DataFrame({"parameter": PARAM_NAMES, "truth": theta_true,
                      "posterior_mean": metric["mean"], "posterior_std": metric["std"],
                      "error_over_prior": metric["normalized_error_prior"],
                      "pull": metric["pull"]}))


In [ ]:
g = plot_corner(samples, theta_true, contract, OUTPUT_DIR)
plt.show()
print("Saved corner plot:", OUTPUT_DIR / "moped_sbi_corner.png")


## What the compression means
The original observable is 40 signed bins of D_ell. The saved MOPED transform first applies asinh(x/s), with per-bin s, mean and standard deviation learned only from optimization rows. It then projects onto nine locally noise-whitened mean-derivative directions and standardizes those nine coordinates with saved training statistics.

The fitted covariance comes from matched noisy-minus-clean spectra in transformed coordinates near Battaglia12, with a fitted parameter-dependent residual mean removed and 5% diagonal shrinkage. It is **not** the covariance of spectra over the whole varying prior, nor the separate 64-seed ensemble covariance. The mean derivatives come from a local quadratic fit in nine prior-normalized parameters.

This is approximate local mean-Fisher compression. The original [MOPED result](https://arxiv.org/abs/astro-ph/9911102) preserves Fisher information under its assumptions, particularly parameter-independent noise covariance. Here the SO cross-spectrum covariance can depend on signal and the response is nonlinear; global losslessness is not claimed. NPE is trained on all in-prior simulations after compression, not on the local fit alone.

The simulator has a fixed halo realization and mask. Changing noise seeds probes noise variability conditional on that sky; it does not add missing cosmological/halo sample variance. Since the legacy training set also reused noise, the fitted residual covariance is not an independent-noise likelihood covariance. Its near-constant directions can amplify new noise dramatically. The saved transform remains unchanged here; changing it without retraining would invalidate the learned NPE. See [the diagnosis](SO_MOPED_FIXED_NOISE_DIAGNOSIS.md) for measured results and limitations.
